In [217]:
import torch
from einops import rearrange, einsum, reduce
from torch import nn

In [218]:
# ALl neural nets modules should inherent from nn.Module parent class -> inherits convenient methods such as: load_state_dict(), to(), get_parameters(), cpu(), cuda(), children(), bfloat16()...

# Implement a Linear Class (= "a Linear Module")
# y = xWT

class Linear(nn.Module): # Inherits nn.Module methods()
    def __init__(self, in_features, out_features, device=None, dtype=None):
        super().__init__()
        
        self.sigma = (2 / (in_features + out_features)) ** (1/2)

        self.weight = nn.Parameter(nn.init.trunc_normal_(torch.empty(out_features, in_features, dtype=dtype, device=device), 
                                                                mean=0, std = self.sigma, 
                                                                a = -3 * self.sigma, b= 3 * self.sigma ))

    def forward(self, x: torch.tensor) -> torch.Tensor: # All nn.Module need to have a forward() method
        return einsum(x, self.weight, '... in_feature , out_feature in_feature -> ... out_feature')

In [219]:
# Create an embedding table Class
# Ounce again, every neural nets module should inherent nn.Module for convenient access to parent methods (.load_state_dict(), .Parameters(), .to()...)

class Embedding(nn.Module):

    def __init__(self, num_embeddings, embedding_dim, dtype=None, device=None):
        super().__init__()
        self.weight = nn.Parameter(nn.init.trunc_normal_(torch.empty(num_embeddings, embedding_dim, dtype=dtype, device=device), std=1, a=-3, b=3))

    def forward(self, x: torch.LongTensor) -> torch.Tensor: # (... T) -> (... T d_model)
        return self.weight[x] # shape: = x.shape + self.weights.shape[1:]

In [220]:
# Implement LayerNorm: RMSNorm
# Dtype: Prevent overflow of root mean square by using dtype = float32. 
# It is possible to change data type in this way: input_dtype -> float32 -> input_dtype

class RMSNorm(nn.Module):

    def __init__(self, d_model, eps=1e-5, dtype=None, device=None):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d_model, dtype=dtype)) # (d_model)
        self.eps = eps

    def forward(self, x): # (b T d_model -> b T d_model)
        in_dtype = x.dtype
        x = x.to(dtype = torch.float32)
        batched_rms = torch.sqrt(reduce(torch.square(x), '... d_model -> ... 1', reduction = 'mean') + self.eps)
        x_norm = torch.div(x, batched_rms)
        result = einsum(x_norm, self.weight, '... d_model, d_model -> ... d_model')
        return result.to(dtype=in_dtype)

In [221]:
# FFN Transformer Layer
# Swish activation function + Gated Linear Unit: W1 & W2, thus d_ffn becomes 2/3 what it would be without gated linear unit to keep same parameter count
# d_model -> d_ffn -> d_model. One single hidden layer. d_ffn = 8/3 * d_model (should be a multiplier of 64 for hardware efficiency)
# The module should - as always - inherent from PyTorch nn.Module parent class for convenient methods usage (.to(), .load_state_dict(), .parameters())
# Weights should be defined inside nn.Parameter() inside the child module for compatibility with Torch other parameter related methods (e.g. .parameters())

class SwiGLU_FFN(nn.Module):
    
    def __init__(self, d_model, dtype=None, device=None):
        super().__init__()
        self.d_ff = int(((8/3) * d_model // 64) * 64)  # Kepping same parameter count with/without Gated Linear Unit
        self.w1 = Linear(in_features=d_model, out_features=self.d_ff, dtype=dtype, device=device)
        self.w2 = Linear(in_features=self.d_ff, out_features=d_model, dtype=dtype, device=device)
        self.w3 = Linear(in_features=d_model, out_features=self.d_ff, dtype=dtype, device=device)

    def forward(self, x): # x.shape == ([B T d_model])
        x_1 = self.w1(x) # ... d_model -> ... d_ff
        GLU = self.w3(x)
        SiLU = einsum(x_1, torch.sigmoid(x_1), '... d_ff, ... d_ff -> ... d_ff')
        SwiGLU = einsum(SiLU, GLU, '... d_ff, ... d_ff -> ... d_ff')
        return self.w2(SwiGLU)

In [222]:
# Relative Positional Embedding (RoPE)
# From (nn.Module). Uses buffer for the rotation angles with 'self.register_buffer()'
# Different dimensions of the latent space get different rotation speed. Rotation itself depends on the position of the token in the sequence.
# In the Latent Space, delta of angle between vectors is linearly proportional to the distance of their index in the sequence

class RoPE(nn.Module):
    # Create a ([T, d_k, d_k]) tensor of the rotation matrices is suboptimal
    # Create a ([T, d_k/2, 2, 2]) tensor of rotation matrices, rearrange x to become (... d_k/2 2) then MatMul then back to (... d_k)
    def __init__(self, theta: int, d_k: int, max_sequence_len: int, device=None):
        super().__init__()
        self.thetas_dim = theta ** (-torch.arange(0, d_k, step=2, device=device) / d_k)
        self.thetas_sequence = einsum(torch.arange(0, max_sequence_len, device=device), self.thetas_dim, 'maxT, d2 -> maxT d2')
        self.stack = torch.stack([torch.cos(self.thetas_sequence), -torch.sin(self.thetas_sequence), 
                                torch.sin(self.thetas_sequence), torch.cos(self.thetas_sequence)]) # (4, maxT, d_k/2)
        self.register_buffer('RoPE', rearrange(self.stack, ' (l c) maxT d2 -> maxT d2 l c', l=2, c=2), persistent=False)
    
    def forward(self, x, token_positions): # x size: (... T d_k) / token_positions size: (... T)
        sequence_rope = self.RoPE[token_positions] # (... T d2 l c)
        x_paired = rearrange(x, '... T (d1 d2) -> ... T d1 d2', d2=2)
        output_paired = einsum(sequence_rope, x_paired, '... T dk2 l c, ... T dk2 c -> ... T dk2 l')
        return rearrange(output_paired, '... T dk2 l -> ... T (dk2 l)')

In [223]:
def Softmax(x: torch.Tensor, dim: int):   
    x_copy = torch.transpose(x, dim, -1) # (... i)
    x_minus_max = reduce(x_copy, '... i -> ... 1', reduction='max')
    x_copy = x_copy - x_minus_max # Stability trick to avoidd exp(vi) to become inf and then having inf/inf = NaN
    x_copy = torch.exp(x_copy)
    x_div = reduce(x_copy, '... i -> ... 1', reduction='sum')
    probs = torch.div(x_copy, x_div)
    return torch.transpose(probs, dim, -1)

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=True):
    T1 = Q.shape[-2]
    T2 = K.shape[-2]
    true_mask = torch.ones(T1,T2, dtype=torch.bool)
    mask_copy = mask * true_mask
    mask_matrix = torch.zeros_like(mask_copy, dtype=torch.float)
    mask_matrix[~mask_copy] = float('-inf')
    d_k = K.shape[-1]
    logits_scores = torch.div(einsum(Q, K, '... T1 d_k, ... T2 d_k -> ... T1 T2'), ( d_k ** (1/2) ))
    mask_matrix = mask_matrix.to(device=Q.device)
    logits_scores += mask_matrix
    scores = Softmax(logits_scores, dim=-1)
    return einsum(scores, V, '... T1 T2, ... T2 d_v -> ... T1 d_v')

In [225]:
Q, K, V = torch.randn(2, 2, 3, 4), torch.randn(2, 2, 3, 4), torch.randn(2, 2, 3, 4)
print('---')

---


In [226]:
print(torch.ones(T,T, dtype=torch.bool))

tensor([[True, True, True, True, True],
        [True, True, True, True, True],
        [True, True, True, True, True],
        [True, True, True, True, True],
        [True, True, True, True, True]])


In [227]:
torch.tril(torch.ones(3, 3, dtype=torch.bool))

tensor([[ True, False, False],
        [ True,  True, False],
        [ True,  True,  True]])

In [228]:
scaled_dot_product_attention(Q, K, V)

tensor([[[[ 0.3189, -0.6013, -0.0084, -0.7611],
          [-1.1356, -1.2398,  0.4617, -0.8844],
          [ 1.2319, -0.9547,  0.3496, -0.9161]],

         [[ 0.4597,  0.4425, -0.3861,  0.3474],
          [-0.5506, -0.0259,  0.1738,  0.6613],
          [-0.4314, -0.4740,  0.3382,  0.2759]]],


        [[[ 0.2647, -0.3222,  0.5966, -0.2324],
          [-1.2088, -0.7623,  0.3527,  0.2269],
          [-0.1713, -2.0061,  0.1859,  0.7232]],

         [[ 0.2485, -0.1054, -0.9042,  1.0868],
          [ 0.2454, -0.0697, -0.9290,  1.0981],
          [ 0.6713, -1.1280, -1.1829,  0.5318]]]])

In [229]:
scaled_dot_product_attention(Q, K, V, torch.tril(torch.ones(3, 3, dtype=torch.bool)))

tensor([[[[ 2.1369,  0.1969, -0.5960, -0.6070],
          [ 2.1368,  0.1969, -0.5960, -0.6070],
          [ 1.2319, -0.9547,  0.3496, -0.9161]],

         [[-0.6709,  0.1240,  0.1462,  0.8411],
          [-0.4662,  0.1986,  0.0421,  0.7634],
          [-0.4314, -0.4740,  0.3382,  0.2759]]],


        [[[ 0.5827, -0.3046,  0.6324, -0.2907],
          [-0.1681, -1.9989,  0.1878,  0.7189],
          [-0.1713, -2.0061,  0.1859,  0.7232]],

         [[ 0.3549, -0.1302, -1.1950,  1.0072],
          [ 0.2445, -0.0676, -0.9282,  1.0993],
          [ 0.6713, -1.1280, -1.1829,  0.5318]]]])

In [230]:
mask = torch.tensor([[True, True, False], [True, False, True], [False, False, True]])

mask_matrix = torch.zeros(mask.shape, dtype=torch.float)
mask_matrix[~mask] = float('-inf')
print(mask_matrix)

tensor([[0., 0., -inf],
        [0., -inf, 0.],
        [-inf, -inf, 0.]])


In [231]:
T = 5

attention_mask = torch.transpose(torch.tril(torch.empty(T, T) - float('inf'), diagonal=-1), dim0=1, dim1=0)

print(attention_mask)

tensor([[0., -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0.]])
